# Random Forest — multibranch-style sequence classifier

This notebook uses the cleaned `base_utils_qwen.py` and trains a sequence-level Random Forest.

Local quick runs use `data/sample.csv` (37 sequences). Set `use_sample_data = False` for full `train.csv`.

Switch `search_mode` between `'grid'` and `'bayesian'` in the config cell.

Style:
- configuration
- data loading
- split
- estimator
- parameter search (grid or Bayesian)
- holdout evaluation
- save results


In [2]:
import os
import sys
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Local path routing (works from notebooks/ or project root)
current_dir = os.getcwd()
workspace_root = current_dir
if os.path.basename(current_dir) == 'notebooks':
    workspace_root = os.path.dirname(current_dir)

src_path = os.path.join(workspace_root, 'src')
sys.path.insert(0, workspace_root)
sys.path.insert(0, src_path)

# Kaggle path routing
try:
    dataset_name = os.listdir('/kaggle/input/datasets/keithmarange')[0]
    sys.path.append(f'/kaggle/input/datasets/keithmarange/{dataset_name}/')
    sys.path.append('/kaggle/input/cmi-competition-code')
except Exception:
    pass

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, GroupKFold, GroupShuffleSplit
from sklearn.metrics import f1_score, make_scorer

try:
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
except ImportError:
    BayesSearchCV = None
    Categorical = Integer = Real = None
    SKOPT_AVAILABLE = False

try:
    from src import data_utils
    from src.base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    print('Imports loaded from src/')
except ImportError:
    import data_utils
    from base_utils_qwen import (
        SequenceExtractor,
        RandomForestSequenceClassifier,
        competition_scorer,
        evaluate_holdout,
        make_competition_scorer,
        prepare_bayesian_space,
    )
    print('Imports loaded from flat src path')


Imports loaded from src/


In [3]:
# Install Bayesian search dependency if missing (safe to re-run)
try:
    import skopt
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-optimize'])
    import skopt
    from skopt import BayesSearchCV
    from skopt.space import Categorical, Integer, Real
    SKOPT_AVAILABLE = True
    print('Installed scikit-optimize')
else:
    print(f'scikit-optimize {skopt.__version__} ready')

scikit-optimize 0.10.2 ready


In [4]:
TARGET_COL = 'bfrb'

# Use data/sample.csv for fast local runs; set False for full train.csv
use_sample_data = True
sample_file = 'sample.csv'  # also available: eg.csv

search_mode = 'grid'  # 'grid' or 'bayesian'
random_state = 42
n_splits = 1          # 1 -> GroupShuffleSplit; >=2 -> GroupKFold
cv_test_size = 0.3
train_size = 0.5
n_iter = 8            # Bayesian iterations (raise for full runs)
verbose = 3

results_dir = Path('results_rf_multibranch_style')
results_dir.mkdir(exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M')

if TARGET_COL == 'bfrb':
    scorer = competition_scorer
else:
    scorer = make_scorer(f1_score, average='macro', zero_division=0)

search_mode = str(search_mode).lower()
if search_mode in ('bayes', 'bayesian'):
    search_mode = 'bayesian'
elif search_mode != 'grid':
    raise ValueError("search_mode must be 'grid' or 'bayesian'")

if n_splits <= 1:
    cv_object = GroupShuffleSplit(
        n_splits=1,
        test_size=cv_test_size,
        random_state=random_state,
    )
else:
    cv_object = GroupKFold(n_splits=n_splits)


In [5]:
data_root = data_utils.find_data_root()
sample_path = data_root / sample_file

if use_sample_data and sample_path.exists():
    raw_train_df = pd.read_csv(sample_path)
    print(f'Using {sample_file}: {raw_train_df["sequence_id"].nunique()} sequences')
else:
    raw_train_df = pd.read_csv(data_root / 'train.csv')
    print(f'Using train.csv: {raw_train_df["sequence_id"].nunique()} sequences')

train_demo_df = pd.read_csv(data_root / 'train_demographics.csv')

train_df = raw_train_df.set_index('row_id').copy(deep=True)

train_df['gesture'] = train_df['gesture'].fillna('non_bfrb').astype(str)
train_df['orientation'] = train_df['orientation'].fillna('Unknown').astype(str)
train_df['is_target'] = train_df['sequence_type'].eq('Target').astype(int)
train_df['bfrb'] = train_df['gesture'].where(train_df['is_target'].astype(bool), 'non_bfrb')

train_df['gesture_position'] = train_df['gesture'].str.split(' - ').str[0]
train_df['gesture_action'] = train_df['gesture'].str.split(' - ').str[-1]

if TARGET_COL not in train_df.columns:
    train_df[TARGET_COL] = train_df['gesture_action']

train_df[TARGET_COL] = train_df[TARGET_COL].fillna('non_bfrb').astype(str)


✅ Found data folder: C:\Users\maran\OneDrive\Documents\Git Profile\cmi_dexter\data
Using sample.csv: 37 sequences


In [6]:
try:
    train_sample_df, hold_out_df = data_utils.sample_balanced_split(
        train_df,
        train_pct=train_size,
        test_pct=min(0.2, 1 - train_size),
        random_state=random_state,
    )
except Exception:
    seq_df = train_df[['sequence_id', 'is_target', TARGET_COL]].drop_duplicates('sequence_id').sort_values('sequence_id')
    gss = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(seq_df, groups=seq_df['sequence_id']))

    train_seqs = seq_df.iloc[train_idx]['sequence_id']
    test_seqs = seq_df.iloc[test_idx]['sequence_id']

    train_sample_df = train_df[train_df['sequence_id'].isin(train_seqs)].copy()
    hold_out_df = train_df[train_df['sequence_id'].isin(test_seqs)].copy()

X_train = train_sample_df.copy()
X_test = hold_out_df.copy()

y_train = X_train[['sequence_id', 'is_target', TARGET_COL]].copy()
y_test = X_test[['sequence_id', 'is_target', TARGET_COL]].copy()

groups = X_train['sequence_id'].astype(str)

print('Train sequences:', X_train['sequence_id'].nunique())
print('Test sequences:', X_test['sequence_id'].nunique())


Train sequences: 18
Test sequences: 19


In [7]:
extractor = SequenceExtractor(
    acc_modes='raw|velocity|jerk',
    rotation_modes='quaternion|angular_velocity',
    tof_modes='pooled_stats|sensor_stats',
    thm_modes='centered_diff',
    motion_filter_mode=None,
    use_dead_reckoning=False,
    compute_dt=True,
    interp_mode='linear',
    padding_value=0.0,
    chunk_window_size=None,
    chunk_stride=None,
    output_format='frame',
    frame_stats='mean,std,min,max,last',
    add_global_context=False,
    resample_modalities=False,
)

estimator = RandomForestClassifier(
    n_estimators=300,
    random_state=random_state,
    n_jobs=-1,
    class_weight='balanced_subsample',
)

model = RandomForestSequenceClassifier(
    primary_target=TARGET_COL,
    extractor=extractor,
    estimator=estimator,
    random_state=random_state,
)


In [8]:
# ============================================================
# PARAMETER SPACE — GRID vs BAYESIAN
# ============================================================

_SKOPT_AVAILABLE = globals().get('SKOPT_AVAILABLE', False)
_ACTIVE_SEARCH_MODE = str(
    globals().get('SEARCH_MODE', globals().get('search_mode', 'grid'))
).lower()
if _ACTIVE_SEARCH_MODE in ('bayes', 'bayesian'):
    _ACTIVE_SEARCH_MODE = 'bayesian'

# Fast grid for sample.csv smoke tests
GRID_PARAM_SPACE_QUICK = {
    'extractor__acc_modes': ['raw|velocity|jerk'],
    'extractor__rotation_modes': ['quaternion|angular_velocity'],
    'extractor__tof_modes': ['pooled_stats|sensor_stats'],
    'extractor__thm_modes': ['centered_diff'],
    'extractor__frame_stats': ['mean,std,min,max,last'],
    'extractor__motion_filter_mode': [None],
    'extractor__use_dead_reckoning': [False],
    'extractor__dead_reckoning_detrend': [False],
    'extractor__kalman_process_noise': [1e-3],
    'extractor__kalman_measurement_noise': [1e-1],
    'extractor__window_size': [7],
    'extractor__smooth_alpha': [None],
    'extractor__clip_value': [None],
    'extractor__interp_mode': ['linear'],
    'extractor__output_format': ['frame'],
    'extractor__padding_value': [0.0],
    'extractor__maxlen': [160],
    'extractor__chunk_window_size': [None],
    'extractor__chunk_stride': [None],
    'extractor__add_global_context': [False],
    'extractor__resample_modalities': [False],
    'extractor__compute_dt': [True],
    'extractor__imu_native_sampling_rate': [20],
    'extractor__rot_native_sampling_rate': [20],
    'extractor__tof_native_sampling_rate': [5],
    'extractor__thm_native_sampling_rate': [5],
    'estimator__n_estimators': [100, 200],
    'estimator__criterion': ['gini'],
    'estimator__max_depth': [None, 20],
    'estimator__min_samples_split': [2],
    'estimator__min_samples_leaf': [1],
    'estimator__max_features': ['sqrt', 0.5],
    'estimator__bootstrap': [True],
    'estimator__class_weight': ['balanced_subsample'],
}

# ============================================================
# GRID SEARCH SPACE
# Practical compact grid. Bayesian space does the full exploration.
# ============================================================

GRID_PARAM_SPACE = {
    # ------------------------------------------------------------
    # EXTRACTOR: main sensor-feature domains
    # ------------------------------------------------------------
    'extractor__acc_modes': [
        'raw|velocity|jerk',
        'smoothed|velocity|displacement|jerk',
    ],

    'extractor__rotation_modes': [
        'quaternion|angular_velocity',
        'quaternion|euler|angular_velocity',
    ],

    'extractor__tof_modes': [
        'pooled_stats|sensor_stats',
        'pooled_stats',
    ],

    'extractor__thm_modes': [
        'centered_diff',
        'centered',
    ],

    'extractor__frame_stats': [
        'mean,std,min,max,last',
        'mean,std,min,max,last,first,rms',
    ],

    # ------------------------------------------------------------
    # EXTRACTOR: fixed preprocessing for stable RF smoke/grid runs
    # ------------------------------------------------------------
    'extractor__motion_filter_mode': [None],
    'extractor__use_dead_reckoning': [False],
    'extractor__dead_reckoning_detrend': [False],
    'extractor__kalman_process_noise': [1e-3],
    'extractor__kalman_measurement_noise': [1e-1],

    'extractor__window_size': [7],
    'extractor__smooth_alpha': [None],
    'extractor__clip_value': [None],
    'extractor__interp_mode': ['linear'],

    # ------------------------------------------------------------
    # EXTRACTOR: fixed frame-output safety params
    # ------------------------------------------------------------
    'extractor__output_format': ['frame'],
    'extractor__padding_value': [0.0],
    'extractor__maxlen': [160],
    'extractor__chunk_window_size': [None],
    'extractor__chunk_stride': [None],
    'extractor__add_global_context': [False],
    'extractor__resample_modalities': [False],
    'extractor__compute_dt': [True],

    'extractor__imu_native_sampling_rate': [20],
    'extractor__rot_native_sampling_rate': [20],
    'extractor__tof_native_sampling_rate': [5],
    'extractor__thm_native_sampling_rate': [5],

    # ------------------------------------------------------------
    # RANDOM FOREST ESTIMATOR
    # ------------------------------------------------------------
    'estimator__n_estimators': [300, 600],
    'estimator__criterion': ['gini'],
    'estimator__max_depth': [None, 30],
    'estimator__min_samples_split': [2],
    'estimator__min_samples_leaf': [1],
    'estimator__max_features': ['sqrt', 0.5],
    'estimator__bootstrap': [True],
    'estimator__class_weight': ['balanced_subsample', 'balanced'],
}


# ============================================================
# BAYESIAN SEARCH SPACE
# Full exploration space.
# ============================================================

if _SKOPT_AVAILABLE:
    try:
        BAYESIAN_PARAM_SPACE = {
            # --------------------------------------------------------
            # EXTRACTOR: sensor-feature domains
            # --------------------------------------------------------
            'extractor__acc_modes': Categorical([
                'raw',
                'raw|velocity',
                'raw|velocity|jerk',
                'raw|velocity|displacement',
                'raw|velocity|displacement|jerk',
                'smoothed|velocity|jerk',
                'smoothed|velocity|displacement|jerk',
            ]),

            'extractor__rotation_modes': Categorical([
                'quaternion',
                'quaternion|euler',
                'quaternion|angular_velocity',
                'quaternion|euler|angular_velocity',
                'quaternion|delta_euler|angular_velocity',
                'quaternion|angular_velocity|delta_euler',
                'quaternion|angular_velocity|delta_euler|rot6d',
            ]),

            'extractor__tof_modes': Categorical([
                'sensor_stats',
                'pooled_stats',
                'pooled_stats|sensor_stats',
            ]),

            'extractor__thm_modes': Categorical([
                'centered',
                'diff',
                'centered_diff',
            ]),

            'extractor__frame_stats': Categorical([
                'mean,std,min,max,last',
                'mean,std,min,max,last,first',
                'mean,std,min,max,last,rms',
                'mean,std,min,max,last,abs_mean',
                'mean,std,min,max,last,median',
                'mean,std,min,max,last,first,rms',
                'mean,std,min,max,last,first,rms,abs_mean',
            ]),

            # --------------------------------------------------------
            # EXTRACTOR: preprocessing / filtering
            # --------------------------------------------------------
            'extractor__motion_filter_mode': Categorical([
                None,
                'kalman',
                'extended_kalman',
            ]),

            'extractor__use_dead_reckoning': Categorical([False, True]),
            'extractor__dead_reckoning_detrend': Categorical([False, True]),

            'extractor__kalman_process_noise': Real(
                1e-5,
                1e-1,
                prior='log-uniform',
            ),

            'extractor__kalman_measurement_noise': Real(
                1e-3,
                1e1,
                prior='log-uniform',
            ),

            'extractor__window_size': Integer(3, 50),

            'extractor__smooth_alpha': Categorical([
                None,
                0.05,
                0.10,
                0.20,
                0.30,
                0.50,
                0.70,
                0.90,
            ]),

            'extractor__clip_value': Categorical([
                None,
                25.0,
                50.0,
                100.0,
                150.0,
            ]),

            'extractor__interp_mode': Categorical([
                'linear',
                'ffill',
            ]),

            # --------------------------------------------------------
            # EXTRACTOR: fixed frame-output safety params
            # --------------------------------------------------------
            'extractor__output_format': Categorical(['frame']),
            'extractor__padding_value': Categorical([0.0]),
            'extractor__maxlen': Categorical([160]),
            'extractor__chunk_window_size': Categorical([None]),
            'extractor__chunk_stride': Categorical([None]),
            'extractor__add_global_context': Categorical([False]),
            'extractor__resample_modalities': Categorical([False]),
            'extractor__compute_dt': Categorical([True]),

            'extractor__imu_native_sampling_rate': Categorical([20]),
            'extractor__rot_native_sampling_rate': Categorical([20]),
            'extractor__tof_native_sampling_rate': Categorical([5]),
            'extractor__thm_native_sampling_rate': Categorical([5]),

            # --------------------------------------------------------
            # RANDOM FOREST ESTIMATOR
            # --------------------------------------------------------
            'estimator__n_estimators': Integer(200, 1200),

            'estimator__criterion': Categorical([
                'gini',
                'entropy',
            ]),

            'estimator__max_depth': Categorical([
                None,
                10,
                20,
                30,
                50,
                80,
            ]),

            'estimator__min_samples_split': Integer(2, 20),
            'estimator__min_samples_leaf': Integer(1, 8),

            'estimator__max_features': Categorical([
                'sqrt',
                'log2',
                0.3,
                0.5,
                0.7,
                0.9,
            ]),

            'estimator__bootstrap': Categorical([True, False]),

            'estimator__class_weight': Categorical([
                'balanced',
                'balanced_subsample',
                None,
            ]),

            'estimator__min_impurity_decrease': Real(0.0, 0.005),
        }

    except Exception:
        BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE
else:
    BAYESIAN_PARAM_SPACE = GRID_PARAM_SPACE


# ============================================================
# ACTIVE PARAM SPACE
# ============================================================

if _ACTIVE_SEARCH_MODE == 'bayesian' and _SKOPT_AVAILABLE:
    param_space = BAYESIAN_PARAM_SPACE
    try:
        param_space = prepare_bayesian_space(param_space)
    except Exception:
        pass
elif use_sample_data:
    param_space = GRID_PARAM_SPACE_QUICK
else:
    param_space = GRID_PARAM_SPACE

if _ACTIVE_SEARCH_MODE == 'grid':
    grid_size = 1
    for values in param_space.values():
        grid_size *= len(values)
    print('Grid combinations:', grid_size)
else:
    print('Bayesian iterations:', n_iter)

print('Active search mode:', _ACTIVE_SEARCH_MODE)
print('Parameter keys:', len(param_space))

Grid combinations: 8
Active search mode: grid
Parameter keys: 34


In [9]:
if search_mode == 'bayesian':
    if not SKOPT_AVAILABLE:
        raise ImportError(
            "Bayesian search requires scikit-optimize. "
            "Install with: pip install scikit-optimize"
        )

    search = BayesSearchCV(
        estimator=model,
        search_spaces=param_space,
        n_iter=n_iter,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        random_state=random_state,
        verbose=verbose,
        return_train_score=True,
        error_score=0.0,
    )
else:
    search = GridSearchCV(
        estimator=model,
        param_grid=param_space,
        scoring=scorer,
        cv=cv_object,
        n_jobs=1,
        verbose=verbose,
        return_train_score=True,
        error_score=0.0,
    )

search.fit(X_train, y_train, groups=groups)

print('Best CV score:', search.best_score_)
print('Best params:', search.best_params_)


Fitting 1 folds for each of 8 candidates, totalling 8 fits
[CV 1/1] END estimator__bootstrap=True, estimator__class_weight=balanced_subsample, estimator__criterion=gini, estimator__max_depth=None, estimator__max_features=sqrt, estimator__min_samples_leaf=1, estimator__min_samples_split=2, estimator__n_estimators=100, extractor__acc_modes=raw|velocity|jerk, extractor__add_global_context=False, extractor__chunk_stride=None, extractor__chunk_window_size=None, extractor__clip_value=None, extractor__compute_dt=True, extractor__dead_reckoning_detrend=False, extractor__frame_stats=mean,std,min,max,last, extractor__imu_native_sampling_rate=20, extractor__interp_mode=linear, extractor__kalman_measurement_noise=0.1, extractor__kalman_process_noise=0.001, extractor__maxlen=160, extractor__motion_filter_mode=None, extractor__output_format=frame, extractor__padding_value=0.0, extractor__resample_modalities=False, extractor__rot_native_sampling_rate=20, extractor__rotation_modes=quaternion|angular_v

In [10]:
best_model = search.best_estimator_

y_pred = best_model.predict(X_test)

eval_results = evaluate_holdout(
    y_test,
    y_pred,
    target_col=TARGET_COL,
    verbose=True,
)

print('Holdout competition score:', eval_results['competition_score'])

cv_df = pd.DataFrame(search.cv_results_)
cv_df.to_csv(results_dir / f'rf_multibranch_style_cv_{timestamp}.csv', index=False)

eval_results['results_df'].to_csv(
    results_dir / f'rf_multibranch_style_holdout_{timestamp}.csv',
    index=False,
)

pd.DataFrame(
    [
        {
            'best_score': search.best_score_,
            'best_params': str(search.best_params_),
            'holdout_score': eval_results['competition_score'],
        }
    ]
).to_csv(
    results_dir / f'rf_multibranch_style_best_{timestamp}.csv',
    index=False,
)



FINAL EVALUATION
Binary F1 (non_bfrb vs bfrb): 0.7368
BFRB Gesture Macro F1: 0.1185
COMPETITION SCORE: 0.4277

----------------------------------------
BFRB Gesture Classification Report
----------------------------------------
                          precision    recall  f1-score   support

   Above ear - pull hair       0.00      0.00      0.00         1
      Cheek - pinch skin       0.00      0.00      0.00         1
     Eyebrow - pull hair       0.00      0.00      0.00         0
     Eyelash - pull hair       0.50      1.00      0.67         1
Forehead - pull hairline       0.50      0.33      0.40         3
      Forehead - scratch       0.00      0.00      0.00         2
       Neck - pinch skin       0.00      0.00      0.00         1
          Neck - scratch       0.00      0.00      0.00         2
                non_bfrb       0.00      0.00      0.00         0

                accuracy                           0.18        11
               macro avg       0.11      0.

In [11]:
importances = pd.Series(
    best_model.estimator_.feature_importances_,
    index=best_model.extractor_.frame_feature_names_,
).sort_values(ascending=False)

print(importances.head(50))


rot_z_angvel_last           0.021000
rot_w_angvel_std            0.015333
acc_y_vel_last              0.014833
tof_3_std_last              0.014833
acc_y_vel_max               0.014333
acc_z_raw_last              0.013045
thm_3_centered_diff_max     0.012690
thm_4_centered_min          0.012167
thm_2_centered_min          0.012167
thm_2_centered_diff_last    0.011712
acc_y_vel_std               0.011546
rot_w_angvel_min            0.011500
thm_1_centered_min          0.011500
ang_vel_mag_mean            0.011476
tof_2_std_max               0.011183
acc_x_jerk_last             0.011049
acc_z_raw_mean              0.011000
rot_w_quat_std              0.010944
acc_z_raw_std               0.010667
acc_z_vel_std               0.010367
rot_x_angvel_max            0.010296
thm_2_centered_mean         0.010167
tof_pooled_std_max          0.010083
acc_x_vel_min               0.010056
rot_y_angvel_min            0.009485
tof_4_std_max               0.009278
rot_x_angvel_mean           0.009277
t